# 01 - Data: Scraping (Fase 1)

## Setup

In [1]:
!pip install -q google-play-scraper==1.2.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import json
import pickle
import shutil
import signal
import time
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from google_play_scraper import Sort, app, reviews

FOLDER_PROJECT = Path('/content/drive/MyDrive/ulasan-ewallet')
FOLDER_RAW = FOLDER_PROJECT / 'data' / 'raw'
FOLDER_SEMENTARA = FOLDER_PROJECT / 'data' / 'sementara'
for folder in [FOLDER_RAW, FOLDER_SEMENTARA]:
    folder.mkdir(parents=True, exist_ok=True)

VERSI_LIBRARY = version('google-play-scraper')
print('google-play-scraper', VERSI_LIBRARY)

# Library mengembalikan waktu tanpa zona waktu, mengikuti zona runtime.
# Konversi ke WIB di bawah mengasumsikan runtime memakai UTC (bawaan Colab).
offset_runtime = datetime.now().astimezone().utcoffset().total_seconds()
assert offset_runtime == 0, 'Runtime tidak memakai UTC, konversi ke WIB perlu disesuaikan'

google-play-scraper 1.2.7


## Konfigurasi

In [5]:
APLIKASI = {
    'dana': 'id.dana',
    'ovo': 'ovo.id',
    'gopay': 'com.gojek.gopay',
}

UKURAN_BATCH = 500             # ulasan per panggilan reviews()
JEDA_DETIK = 2                 # jeda antar batch
MAKS_COBA_ULANG = 3            # percobaan per batch sebelum menyerah
JEDA_COBA_ULANG_DETIK = 15
SIMPAN_SETIAP_BATCH = 10       # frekuensi checkpoint
BATAS_WAKTU_BATCH_DETIK = 120  # jaga-jaga kalau satu request macet

ZONA_WAKTU = 'Asia/Jakarta'

PEMETAAN_KOLOM = {
    'reviewId': 'id_ulasan',
    'content': 'teks',
    'score': 'rating',
    'thumbsUpCount': 'jumlah_suka',
    'reviewCreatedVersion': 'versi',
    'at': 'tanggal',
    'replyContent': 'balasan',
    'repliedAt': 'tanggal_balasan',
}
KOLOM_STANDAR = ['aplikasi'] + list(PEMETAAN_KOLOM.values())

## Verifikasi ID aplikasi

In [6]:
daftar_info = []
for nama, id_app in APLIKASI.items():
    try:
        info = app(id_app, lang='id', country='id')
        daftar_info.append({
            'aplikasi': nama,
            'id_app': id_app,
            'judul': info.get('title'),
            'developer': info.get('developer'),
            'jumlah_install': info.get('installs'),
            'jumlah_rating': info.get('ratings'),
            'jumlah_ulasan': info.get('reviews'),
            'skor': info.get('score'),
            'update_terakhir': info.get('lastUpdatedOn'),
        })
    except Exception as e:
        daftar_info.append({'aplikasi': nama, 'id_app': id_app, 'judul': f'GAGAL: {e}'})

pd.DataFrame(daftar_info)

,aplikasi,id_app,judul,developer,jumlah_install,jumlah_rating,jumlah_ulasan,skor,update_terakhir
0,dana,id.dana,DANA Dompet Digital Indonesia,PT Espay Debit Indonesia Koe,100.000.000+,10864240,2399558,4.644347,17 Okt 2026
1,ovo,ovo.id,OVO,PT Visionet Internasional,50.000.000+,1202525,577062,3.400487,9 Okt 2026
2,gopay,com.gojek.gopay,"GoPay: Transfer, Bayar, QRIS",PT. GoTo Gojek Tokopedia Tbk,100.000.000+,2059725,497867,4.685703,15 Okt 2026


## Fungsi scraping

In [7]:
class WaktuHabis(BaseException):
    """Turunan BaseException agar tidak tertangkap `except Exception` di dalam library."""


def _alarm_waktu_habis(signum, frame):
    raise WaktuHabis(f'request lebih dari {BATAS_WAKTU_BATCH_DETIK} detik')


signal.signal(signal.SIGALRM, _alarm_waktu_habis)


def ke_dataframe(hasil_mentah, nama):
    """Ubah hasil reviews() ke kolom standar. userName dan userImage tidak ikut diambil."""
    if not hasil_mentah:
        return pd.DataFrame(columns=KOLOM_STANDAR)
    df = pd.DataFrame(hasil_mentah)[list(PEMETAAN_KOLOM)].rename(columns=PEMETAAN_KOLOM)
    df.insert(0, 'aplikasi', nama)
    for kolom in ['tanggal', 'tanggal_balasan']:
        df[kolom] = (
            pd.to_datetime(df[kolom])
            .dt.tz_localize('UTC')
            .dt.tz_convert(ZONA_WAKTU)
            .dt.tz_localize(None)
        )
    return df


def ambil_satu_batch(id_app, token):
    """Satu panggilan reviews() dengan coba ulang dan batas waktu."""
    for percobaan in range(1, MAKS_COBA_ULANG + 1):
        signal.alarm(BATAS_WAKTU_BATCH_DETIK)
        try:
            hasil = reviews(
                id_app,
                lang='id',
                country='id',
                sort=Sort.NEWEST,
                count=UKURAN_BATCH,
                continuation_token=token,
            )
            signal.alarm(0)
            return hasil
        except (Exception, WaktuHabis) as e:
            signal.alarm(0)
            print(f'  gagal (percobaan {percobaan}/{MAKS_COBA_ULANG}): {e}')
            if percobaan == MAKS_COBA_ULANG:
                raise
            time.sleep(JEDA_COBA_ULANG_DETIK)


def scrape_aplikasi(nama, id_app, batas_tanggal=None, maks_ulasan=None, folder_checkpoint=None):
    """
    Ambil ulasan dari yang terbaru ke terlama sampai salah satu kondisi berhenti terpenuhi.
    Kalau folder_checkpoint diisi, data disimpan per SIMPAN_SETIAP_BATCH batch beserta
    continuation token, dan pemanggilan berikutnya melanjutkan dari checkpoint terakhir.
    Mengembalikan (DataFrame, info).
    """
    bagian = []     # DataFrame yang sudah tersimpan di checkpoint
    penampung = []  # DataFrame yang belum tersimpan
    token = None

    if folder_checkpoint is not None:
        folder_checkpoint.mkdir(parents=True, exist_ok=True)
        path_token = folder_checkpoint / 'token.pkl'
        file_bagian = sorted(folder_checkpoint.glob('bagian_*.parquet'))
        if file_bagian and path_token.exists():
            bagian = [pd.read_parquet(p) for p in file_bagian]
            with open(path_token, 'rb') as f:
                token = pickle.load(f)
            print(f'[{nama}] lanjut dari checkpoint: {sum(map(len, bagian)):,} ulasan')

    jumlah = sum(map(len, bagian))
    jumlah_batch = 0
    total_detik_request = 0.0
    waktu_mulai = time.time()
    alasan_berhenti = None

    while alasan_berhenti is None:
        t0 = time.time()
        hasil, token = ambil_satu_batch(id_app, token)
        total_detik_request += time.time() - t0
        jumlah_batch += 1

        df_batch = ke_dataframe(hasil, nama)
        if len(df_batch):
            penampung.append(df_batch)
            jumlah += len(df_batch)

        if len(df_batch) == 0 or token.token is None:
            alasan_berhenti = 'scraper tidak mengembalikan data lagi'
        elif batas_tanggal is not None and df_batch['tanggal'].min() < batas_tanggal:
            alasan_berhenti = f'batas tanggal {batas_tanggal:%Y-%m-%d} tercapai'
        elif maks_ulasan is not None and jumlah >= maks_ulasan:
            alasan_berhenti = f'batas jumlah {maks_ulasan:,} tercapai'

        waktunya_simpan = jumlah_batch % SIMPAN_SETIAP_BATCH == 0 or alasan_berhenti is not None
        if folder_checkpoint is not None and penampung and waktunya_simpan:
            df_bagian = pd.concat(penampung, ignore_index=True)
            df_bagian.to_parquet(folder_checkpoint / f'bagian_{len(bagian) + 1:05d}.parquet', index=False)
            with open(path_token, 'wb') as f:
                pickle.dump(token, f)
            bagian.append(df_bagian)
            penampung = []

        if jumlah_batch % SIMPAN_SETIAP_BATCH == 0:
            terlama = df_batch['tanggal'].min() if len(df_batch) else '-'
            menit = (time.time() - waktu_mulai) / 60
            print(f'[{nama}] {jumlah:,} ulasan | terlama {terlama} | {menit:.1f} menit')

        if alasan_berhenti is None:
            time.sleep(JEDA_DETIK)

    semua = bagian + penampung
    df = pd.concat(semua, ignore_index=True) if semua else pd.DataFrame(columns=KOLOM_STANDAR)
    info = {
        'aplikasi': nama,
        'id_app': id_app,
        'jumlah_baris': len(df),
        'tanggal_terlama': str(df['tanggal'].min()),
        'tanggal_terbaru': str(df['tanggal'].max()),
        'jumlah_batch_sesi_ini': jumlah_batch,
        'durasi_menit_sesi_ini': round((time.time() - waktu_mulai) / 60, 1),
        'detik_per_batch': round(total_detik_request / jumlah_batch + JEDA_DETIK, 2),
        'alasan_berhenti': alasan_berhenti,
    }
    print(f'[{nama}] selesai: {len(df):,} ulasan, {alasan_berhenti}')
    return df, info

## Uji kecil

In [8]:
UKURAN_UJI = 1_000

hasil_uji = []
ringkasan_uji = []
for nama, id_app in APLIKASI.items():
    df, info = scrape_aplikasi(nama, id_app, maks_ulasan=UKURAN_UJI)
    hasil_uji.append(df)

    rentang_hari = (df['tanggal'].max() - df['tanggal'].min()).total_seconds() / 86_400
    ulasan_per_hari = len(df) / rentang_hari
    baris = {
        'aplikasi': nama,
        'jumlah': len(df),
        'ulasan_terlama': df['tanggal'].min(),
        'ulasan_terbaru': df['tanggal'].max(),
        'rentang_hari': round(rentang_hari, 2),
        'ulasan_per_hari': round(ulasan_per_hari),
        'detik_per_batch': info['detik_per_batch'],
        'persen_versi_kosong': round(df['versi'].fillna('').eq('').mean() * 100, 1),
        'persen_dibalas': round(df['balasan'].notna().mean() * 100, 1),
    }
    for bulan in [12, 24]:
        estimasi_ulasan = ulasan_per_hari * 365 / 12 * bulan
        estimasi_jam = estimasi_ulasan / UKURAN_BATCH * info['detik_per_batch'] / 3600
        baris[f'estimasi_ulasan_{bulan}bln'] = round(estimasi_ulasan)
        baris[f'estimasi_jam_{bulan}bln'] = round(estimasi_jam, 1)
    ringkasan_uji.append(baris)

df_uji = pd.concat(hasil_uji, ignore_index=True)
df_uji.to_parquet(FOLDER_SEMENTARA / 'uji_kecil.parquet', index=False)

display(pd.DataFrame(ringkasan_uji).set_index('aplikasi').T)
print('Distribusi rating (%):')
display(pd.crosstab(df_uji['aplikasi'], df_uji['rating'], normalize='index').mul(100).round(1))

[dana] selesai: 1,000 ulasan, batas jumlah 1,000 tercapai
[ovo] selesai: 1,000 ulasan, batas jumlah 1,000 tercapai
[gopay] selesai: 1,000 ulasan, batas jumlah 1,000 tercapai


aplikasi,dana,ovo,gopay
jumlah,1000,1000,1000
ulasan_terlama,2026-10-16 01:17:38,2026-09-30 20:27:21,2026-10-15 20:08:00
ulasan_terbaru,2026-10-17 14:01:20,2026-10-17 13:59:57,2026-10-17 14:00:22
rentang_hari,1.53,16.73,1.74
ulasan_per_hari,653,60,573
detik_per_batch,2.42,2.24,2.93
persen_versi_kosong,15.2,28.2,18.4
persen_dibalas,28.9,1.3,19.9
estimasi_ulasan_12bln,238508,21816,209205
estimasi_jam_12bln,0.3,0.0,0.3


Distribusi rating (%):


rating,1,2,3,4,5
aplikasi,,,,,
dana,19.1,4.9,3.2,6.5,66.3
gopay,17.4,2.4,3.0,5.3,71.9
ovo,64.5,5.3,3.6,2.5,24.1


## Scraping penuh

In [9]:
# Periode 12 bulan penuh, disepakati dari hasil uji kecil.
TANGGAL_MULAI = pd.Timestamp('2025-10-01')
TANGGAL_SELESAI = pd.Timestamp('2026-09-30')

assert TANGGAL_MULAI is not None and TANGGAL_SELESAI is not None, 'Isi periode dulu setelah uji kecil dibahas'

for nama, id_app in APLIKASI.items():
    path_final = FOLDER_RAW / f'{nama}.parquet'
    if path_final.exists():
        print(f'[{nama}] file final sudah ada, dilewati')
        continue

    folder_checkpoint = FOLDER_SEMENTARA / nama
    df, info = scrape_aplikasi(nama, id_app, batas_tanggal=TANGGAL_MULAI, folder_checkpoint=folder_checkpoint)
    if df['tanggal'].min() > TANGGAL_MULAI:
        print(f'[{nama}] PERINGATAN: data berhenti di {df["tanggal"].min()}, belum mencapai TANGGAL_MULAI')

    info['tanggal_scraping'] = pd.Timestamp.now(tz=ZONA_WAKTU).strftime('%Y-%m-%d %H:%M WIB')
    df.to_parquet(path_final, index=False)
    path_info = FOLDER_RAW / f'{nama}_info.json'
    path_info.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding='utf-8')
    shutil.rmtree(folder_checkpoint)  # checkpoint tidak diperlukan lagi setelah file final tersimpan

[dana] 5,000 ulasan | terlama 2026-10-10 13:08:01 | 0.4 menit
[dana] 10,000 ulasan | terlama 2026-10-04 09:15:05 | 0.7 menit
[dana] 15,000 ulasan | terlama 2026-10-01 21:53:50 | 1.2 menit
[dana] 20,000 ulasan | terlama 2026-10-01 19:30:24 | 1.6 menit
[dana] 25,000 ulasan | terlama 2026-09-26 07:02:27 | 2.0 menit
[dana] 30,000 ulasan | terlama 2026-09-25 22:07:28 | 2.5 menit
[dana] 35,000 ulasan | terlama 2026-09-25 20:48:55 | 2.9 menit
[dana] 40,000 ulasan | terlama 2026-09-20 20:06:23 | 3.4 menit
[dana] 45,000 ulasan | terlama 2026-09-14 12:47:35 | 3.8 menit
[dana] 50,000 ulasan | terlama 2026-09-08 05:05:04 | 4.2 menit
[dana] 55,000 ulasan | terlama 2026-09-02 11:33:46 | 4.7 menit
[dana] 60,000 ulasan | terlama 2026-08-26 19:25:25 | 5.2 menit
[dana] 65,000 ulasan | terlama 2026-08-20 17:31:46 | 5.6 menit
[dana] 70,000 ulasan | terlama 2026-08-14 22:39:55 | 6.1 menit
[dana] 75,000 ulasan | terlama 2026-08-08 22:48:59 | 6.6 menit
[dana] 80,000 ulasan | terlama 2026-08-02 11:36:14 | 7.1

## CATATAN.md

In [10]:
def tulis_catatan():
    baris = [
        '# Catatan scraping',
        '',
        f'- Library: google-play-scraper {VERSI_LIBRARY}',
        f'- Parameter: lang=id, country=id, sort=NEWEST, batch {UKURAN_BATCH}, jeda {JEDA_DETIK} detik',
        f'- Periode analisis: {TANGGAL_MULAI:%Y-%m-%d} s.d. {TANGGAL_SELESAI:%Y-%m-%d} (filter di Fase 2)',
        '- Kolom tanggal dan tanggal_balasan dalam WIB (dikonversi dari UTC runtime Colab)',
        '- userName dan userImage tidak diambil',
        '',
        '| aplikasi | ID | tanggal scraping | jumlah baris | ulasan terlama | ulasan terbaru | alasan berhenti |',
        '|---|---|---|---|---|---|---|',
    ]
    for nama in APLIKASI:
        path_info = FOLDER_RAW / f'{nama}_info.json'
        if not path_info.exists():
            continue
        info = json.loads(path_info.read_text(encoding='utf-8'))
        baris.append(
            f"| {info['aplikasi']} | `{info['id_app']}` | {info['tanggal_scraping']} | {info['jumlah_baris']:,} "
            f"| {info['tanggal_terlama']} | {info['tanggal_terbaru']} | {info['alasan_berhenti']} |"
        )
    (FOLDER_RAW / 'CATATAN.md').write_text('\n'.join(baris) + '\n', encoding='utf-8')
    print('\n'.join(baris))


tulis_catatan()

# Catatan scraping

- Library: google-play-scraper 1.2.7
- Parameter: lang=id, country=id, sort=NEWEST, batch 500, jeda 2 detik
- Periode analisis: 2025-10-01 s.d. 2026-09-30 (filter di Fase 2)
- Kolom tanggal dan tanggal_balasan dalam WIB (dikonversi dari UTC runtime Colab)
- userName dan userImage tidak diambil

| aplikasi | ID | tanggal scraping | jumlah baris | ulasan terlama | ulasan terbaru | alasan berhenti |
|---|---|---|---|---|---|---|
| dana | `id.dana` | 2026-10-18 14:44 WIB | 347,500 | 2025-09-30 19:37:21 | 2026-10-17 14:01:20 | batas tanggal 2025-10-01 tercapai |
| ovo | `ovo.id` | 2026-10-18 14:46 WIB | 27,000 | 2025-09-25 17:08:31 | 2026-10-17 14:35:59 | batas tanggal 2025-10-01 tercapai |
| gopay | `com.gojek.gopay` | 2026-10-18 15:06 WIB | 205,000 | 2025-09-30 12:28:35 | 2026-10-17 14:35:54 | batas tanggal 2025-10-01 tercapai |


## Validasi

In [11]:
df_raw = pd.concat(
    [pd.read_parquet(FOLDER_RAW / f'{nama}.parquet') for nama in APLIKASI],
    ignore_index=True,
)

ringkasan = df_raw.groupby('aplikasi').agg(
    jumlah=('id_ulasan', 'size'),
    id_duplikat=('id_ulasan', lambda s: int(s.duplicated().sum())),
    ulasan_terlama=('tanggal', 'min'),
    ulasan_terbaru=('tanggal', 'max'),
    rating_rata2=('rating', lambda s: round(s.mean(), 2)),
    persen_versi_kosong=('versi', lambda s: round(s.fillna('').eq('').mean() * 100, 1)),
    persen_dibalas=('balasan', lambda s: round(s.notna().mean() * 100, 1)),
)
display(ringkasan)

print('Distribusi rating (%):')
display(pd.crosstab(df_raw['aplikasi'], df_raw['rating'], normalize='index').mul(100).round(1))

print('Jumlah ulasan per bulan:')
display(pd.crosstab(df_raw['tanggal'].dt.to_period('M'), df_raw['aplikasi']))

,jumlah,id_duplikat,ulasan_terlama,ulasan_terbaru,rating_rata2,persen_versi_kosong,persen_dibalas
aplikasi,,,,,,,
dana,347500,0,2025-09-30 19:37:21,2026-10-17 14:01:20,3.85,17.1,27.6
gopay,205000,0,2025-09-30 12:28:35,2026-10-17 14:35:54,4.21,17.1,15.6
ovo,27000,0,2025-09-25 17:08:31,2026-10-17 14:35:59,2.26,27.7,1.6


Distribusi rating (%):


rating,1,2,3,4,5
aplikasi,,,,,
dana,22.0,4.0,4.5,6.1,63.4
gopay,14.8,2.4,3.3,6.6,72.9
ovo,61.0,6.1,4.5,3.1,25.3


Jumlah ulasan per bulan:


aplikasi,dana,gopay,ovo
tanggal,,,
2025-09,228,363,421
2025-10,24818,17203,2614
2025-11,24366,16082,2294
2025-12,27614,16075,2338
2026-01,32315,18003,2709
2026-02,26940,13409,1918
2026-03,29698,15946,1773
2026-04,22285,12986,2129
2026-05,23432,14781,1912


# Fase 2 - Cleaning, sampling & preprocessing

In [12]:
FOLDER_PROCESSED = FOLDER_PROJECT / 'data' / 'processed'
FOLDER_PROCESSED.mkdir(parents=True, exist_ok=True)

# Periode analisis (WIB), sama dengan batas scraping
TANGGAL_MULAI = pd.Timestamp('2025-10-01')
TANGGAL_SELESAI = pd.Timestamp('2026-09-30')
SEED = 42

## 2.1 Filter periode, duplikat, dan ulasan kosong

In [13]:
df_raw = pd.concat(
    [pd.read_parquet(FOLDER_RAW / f'{nama}.parquet') for nama in APLIKASI],
    ignore_index=True,
)

jejak = []  # jumlah baris per aplikasi setelah tiap langkah cleaning


def catat(langkah, data):
    jejak[:] = [j for j in jejak if j['langkah'] != langkah]  # aman kalau sel dijalankan ulang
    jejak.append({'langkah': langkah, **data.groupby('aplikasi').size().to_dict(), 'total': len(data)})


catat('raw', df_raw)

dalam_periode = (df_raw['tanggal'] >= TANGGAL_MULAI) & (df_raw['tanggal'] < TANGGAL_SELESAI + pd.Timedelta(days=1))
df = df_raw[dalam_periode].copy()
catat('dalam periode', df)

df = df.drop_duplicates('id_ulasan')
catat('tanpa duplikat id', df)

df['teks'] = df['teks'].fillna('').str.strip()
df = df[df['teks'] != '']
catat('teks tidak kosong', df)

pd.DataFrame(jejak).set_index('langkah')

,dana,gopay,ovo,total
langkah,,,,
raw,347500,205000,27000,579500
dalam periode,326378,194383,25591,546352
tanpa duplikat id,326378,194383,25591,546352
teks tidak kosong,326378,194383,25591,546352


## 2.2 Cek lonjakan volume

In [14]:
# Kunci pembanding teks: huruf kecil, tanpa tanda baca, spasi dirapikan
df['teks_kunci'] = (
    df['teks'].str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
# Teks kembar = teks yang persis sama muncul lebih dari sekali di aplikasi yang sama
df['teks_kembar'] = df.duplicated(['aplikasi', 'teks_kunci'], keep=False)
df['hari'] = df['tanggal'].dt.normalize()
df['jumlah_kata'] = df['teks'].str.split().str.len()

harian = df.groupby(['aplikasi', 'hari']).size().rename('jumlah').reset_index()
harian['median_aplikasi'] = harian.groupby('aplikasi')['jumlah'].transform('median')
harian['rasio'] = (harian['jumlah'] / harian['median_aplikasi']).round(1)

BATAS_LONJAKAN = 3
hari_lonjakan = harian[harian['rasio'] > BATAS_LONJAKAN]
df['hari_lonjakan'] = pd.MultiIndex.from_frame(df[['aplikasi', 'hari']]).isin(
    pd.MultiIndex.from_frame(hari_lonjakan[['aplikasi', 'hari']])
)

print('Median ulasan per hari:', harian.groupby('aplikasi')['jumlah'].median().to_dict())
print(f'Hari lonjakan (> {BATAS_LONJAKAN}x median harian aplikasinya):')
display(hari_lonjakan.sort_values(['aplikasi', 'hari']))

Median ulasan per hari: {'dana': 834.0, 'gopay': 519.0, 'ovo': 61.0}
Hari lonjakan (> 3x median harian aplikasinya):


,aplikasi,hari,jumlah,median_aplikasi,rasio
109,dana,2026-01-18,4396,834.0,5.3
292,dana,2026-07-20,5433,834.0,6.5
359,dana,2026-09-25,11591,834.0,13.9
665,gopay,2026-07-28,4295,519.0,8.3
735,ovo,2025-10-06,309,61.0,5.1
736,ovo,2025-10-07,258,61.0,4.2
832,ovo,2026-01-11,446,61.0,7.3
930,ovo,2026-04-19,187,61.0,3.1
1041,ovo,2026-08-08,266,61.0,4.4


In [15]:
df['bintang1'] = df['rating'].eq(1)
df['bintang5'] = df['rating'].eq(5)
df['jam'] = df['tanggal'].dt.hour


def ringkas(data, kunci):
    hasil = data.groupby(kunci).agg(
        jumlah=('id_ulasan', 'size'),
        persen_bintang1=('bintang1', 'mean'),
        persen_bintang5=('bintang5', 'mean'),
        persen_teks_kembar=('teks_kembar', 'mean'),
        median_kata=('jumlah_kata', 'median'),
    )
    kolom_persen = ['persen_bintang1', 'persen_bintang5', 'persen_teks_kembar']
    hasil[kolom_persen] = (hasil[kolom_persen] * 100).round(1)
    return hasil


print('Hari normal (pembanding):')
display(ringkas(df[~df['hari_lonjakan']], 'aplikasi'))

print('Hari lonjakan:')
profil = ringkas(df[df['hari_lonjakan']], ['aplikasi', 'hari'])
per_jam = df[df['hari_lonjakan']].groupby(['aplikasi', 'hari', 'jam']).size()
profil['jam_tersibuk'] = per_jam.groupby(level=[0, 1]).idxmax().str[2]
profil['persen_di_jam_tersibuk'] = (per_jam.groupby(level=[0, 1]).max() / profil['jumlah'] * 100).round(1)
display(profil)

# Angka panjang (nomor HP/rekening) di-masking sebelum teks ditampilkan
print('Teks paling sering muncul di hari lonjakan (10 teratas per aplikasi):')
teks_teratas = (
    df[df['hari_lonjakan']]
    .groupby(['aplikasi', 'teks_kunci']).size()
    .rename('jumlah').reset_index()
    .sort_values('jumlah', ascending=False)
    .groupby('aplikasi').head(10)
)
teks_teratas['teks_kunci'] = teks_teratas['teks_kunci'].str.replace(r'\d{6,}', '<angka>', regex=True)
display(teks_teratas)

Hari normal (pembanding):


,jumlah,persen_bintang1,persen_bintang5,persen_teks_kembar,median_kata
aplikasi,,,,,
dana,304958,18.3,67.7,48.3,3.0
gopay,190088,13.6,74.3,51.4,3.0
ovo,24125,59.7,26.3,15.1,10.0


Hari lonjakan:


jumlah  persen_bintang1  persen_bintang5  \
aplikasi hari                                                   
dana     2026-01-18    4396             50.4             29.3   
         2026-07-20    5433             60.1             20.8   
         2026-09-25   11591             61.4             18.0   
gopay    2026-07-28    4295             59.1             19.1   
ovo      2025-10-06     309             83.5              7.4   
         2025-10-07     258             76.0             10.9   
         2026-01-11     446             75.6             10.5   
         2026-04-19     187             77.0              9.6   
         2026-08-08     266             84.2              6.8   

                     persen_teks_kembar  median_kata  jam_tersibuk  \
aplikasi hari                                                        
dana     2026-01-18                23.8          7.0            22   
         2026-07-20                20.1          8.0            12   
         2026-09-25                15.4          8.0            22   
gopay    2026-07-28                16.2          7.0            15   
ovo      2025-10-06                 3.9         17.0            23   
         2025-10-07                 3.5         15.0             0   
         2026-01-11                 6.3         13.0             6   
         2026-04-19                 7.5         16.0            19   
         2026-08-08                 3.4         16.5             0   

                     persen_di_jam_tersibuk  
aplikasi hari                                
dana     2026-01-18                    25.5  
         2026-07-20                    26.3  
         2026-09-25                    34.8  
gopay    2026-07-28                    20.4  
ovo      2025-10-06                    74.4  
         2025-10-07                    38.8  
         2026-01-11                    12.3  
         2026-04-19                     8.6  
         2026-08-08                    33.5

Teks paling sering muncul di hari lonjakan (10 teratas per aplikasi):


,aplikasi,teks_kunci,jumlah
3061,dana,bagus,316
12314,dana,mantap,251
13383,dana,ok,158
3252,dana,baik,108
14503,dana,sangat membantu,104
15737,dana,sering gangguan,83
14375,dana,sangat bagus,82
14413,dana,sangat baik,64
7153,dana,gangguan terus,56
7628,dana,good,55


### Keputusan hari lonjakan: dipertahankan

## 2.3 Filter jumlah kata minimal

In [16]:
import re


def samarkan(teks):
    """Masking data pribadi sebelum teks ditampilkan: email dan angka panjang (no HP/rekening)."""
    teks = re.sub(r'\S+@\S+', '<email>', teks)
    return re.sub(r'\d{6,}', '<angka>', teks)


MIN_KATA = 2  # rencana awal 3; lihat simulasi di bawah

print('Persen ulasan yang terbuang menurut ambang:')
display(pd.DataFrame({
    f'< {k} kata': (df['jumlah_kata'] < k).groupby(df['aplikasi']).mean().mul(100).round(1)
    for k in [2, 3, 4]
}))

for k in [1, 2]:
    print(f'Teks {k} kata paling sering (10 teratas per aplikasi):')
    teratas = (
        df[df['jumlah_kata'] == k]
        .groupby(['aplikasi', 'teks_kunci']).size()
        .rename('jumlah').reset_index()
        .sort_values('jumlah', ascending=False)
        .groupby('aplikasi').head(10)
    )
    teratas['teks_kunci'] = teratas['teks_kunci'].map(samarkan)
    display(teratas)

terbuang = df['jumlah_kata'] < MIN_KATA
status = terbuang.map({True: 'dibuang', False: 'dipakai'})
print(f'Distribusi rating (%), ulasan < {MIN_KATA} kata vs sisanya:')
display(pd.crosstab([df['aplikasi'], status], df['rating'], normalize='index').mul(100).round(1))

df = df[~terbuang].copy()
catat(f'>= {MIN_KATA} kata', df)
pd.DataFrame(jejak).set_index('langkah')

Persen ulasan yang terbuang menurut ambang:


,< 2 kata,< 3 kata,< 4 kata
aplikasi,,,
dana,26.4,42.6,49.6
gopay,30.0,48.1,56.8
ovo,10.6,17.4,21.8


Teks 1 kata paling sering (10 teratas per aplikasi):


,aplikasi,teks_kunci,jumlah
207,dana,bagus,17707
1781,dana,mantap,15369
3628,gopay,bagus,11261
4857,gopay,mantap,11048
2332,dana,ok,9078
332,dana,baik,6274
5320,gopay,ok,5333
945,dana,good,4388
4166,gopay,good,3494
0,dana,,3401


Teks 2 kata paling sering (10 teratas per aplikasi):


,aplikasi,teks_kunci,jumlah
7620,dana,sangat membantu,7705
7361,dana,sangat bagus,5647
7391,dana,sangat baik,3792
16043,gopay,sangat bagus,3486
16227,gopay,sangat membantu,3087
7817,dana,sangat puas,2081
16072,gopay,sangat baik,1839
1138,dana,bagus sekali,1073
16360,gopay,sangat puas,1036
7680,dana,sangat memuaskan,1028


Distribusi rating (%), ulasan < 2 kata vs sisanya:


rating                   1    2    3    4     5
aplikasi jumlah_kata                           
dana     dibuang       2.6  0.6  2.0  6.8  88.0
         dipakai      27.5  5.0  5.3  6.0  56.2
gopay    dibuang       2.2  0.6  2.3  7.4  87.5
         dipakai      19.9  3.1  3.7  6.3  67.0
ovo      dibuang       8.8  1.0  2.2  5.3  82.6
         dipakai      66.9  6.8  4.9  2.9  18.5

,dana,gopay,ovo,total
langkah,,,,
raw,347500,205000,27000,579500
dalam periode,326378,194383,25591,546352
tanpa duplikat id,326378,194383,25591,546352
teks tidak kosong,326378,194383,25591,546352
>= 2 kata,240075,136038,22890,399003


## 2.4 Deteksi bahasa

In [17]:
!pip install -q fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [18]:
import urllib.request

import fasttext

PATH_MODEL_BAHASA = Path('/content/lid.176.bin')
if not PATH_MODEL_BAHASA.exists():
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin', PATH_MODEL_BAHASA
    )
model_bahasa = fasttext.load_model(str(PATH_MODEL_BAHASA))


def deteksi_bahasa(teks):
    """(kode bahasa, probabilitas) untuk satu teks."""
    prediksi = model_bahasa.f.predict(teks.replace('\n', ' '), 1, 0.0, 'replace')
    if not prediksi:
        return 'tidak_diketahui', 0.0
    prob, label = prediksi[0]
    return label.replace('__label__', ''), prob


hasil = [deteksi_bahasa(t) for t in df['teks'].str.lower()]
df['bahasa'] = [b for b, _ in hasil]
df['prob_bahasa'] = [round(p, 3) for _, p in hasil]

print('Bahasa terdeteksi (% per aplikasi), 8 teratas:')
display(
    pd.crosstab(df['bahasa'], df['aplikasi'], normalize='columns')
    .mul(100).round(2)
    .sort_values('dana', ascending=False)
    .head(8)
)

df['kelompok_kata'] = pd.cut(df['jumlah_kata'], bins=[0, 2, 5, 10, 10_000], labels=['1-2', '3-5', '6-10', '>10'])
print('Persen terdeteksi bukan Indonesia menurut panjang ulasan:')
display(
    (df['bahasa'] != 'id').groupby([df['kelompok_kata'], df['aplikasi']], observed=True).mean()
    .mul(100).round(1).unstack()
)

ke_id = df['bahasa'] == 'id'
print('Persen label id dengan probabilitas < 0,5:', round((df.loc[ke_id, 'prob_bahasa'] < 0.5).mean() * 100, 1))

jumlah_lain = df.loc[~ke_id, 'bahasa'].value_counts()
bahasa_lain = jumlah_lain[jumlah_lain >= 8].head(5).index
contoh = df[df['bahasa'].isin(bahasa_lain)].groupby('bahasa').sample(n=8, random_state=SEED)
contoh = contoh.assign(teks=contoh['teks'].map(samarkan).str.slice(0, 150))
print('Contoh ulasan yang terdeteksi bukan Indonesia (8 per bahasa):')
with pd.option_context('display.max_colwidth', 150):
    display(contoh[['aplikasi', 'bahasa', 'prob_bahasa', 'rating', 'teks']])

Bahasa terdeteksi (% per aplikasi), 8 teratas:


aplikasi,dana,gopay,ovo
bahasa,,,
id,82.84,76.52,81.32
en,4.85,8.44,4.36
ms,4.12,5.02,7.84
tr,1.40,1.47,0.37
sv,0.88,1.37,0.35
pl,0.73,0.99,0.98
it,0.66,0.74,0.73
fr,0.34,0.26,0.14


Persen terdeteksi bukan Indonesia menurut panjang ulasan:


aplikasi,dana,gopay,ovo
kelompok_kata,,,
1-2,35.1,43.1,52.4
3-5,23.4,27.5,36.7
6-10,11.2,13.7,21.7
>10,3.8,5.6,8.4


Persen label id dengan probabilitas < 0,5: 16.4
Contoh ulasan yang terdeteksi bukan Indonesia (8 per bahasa):


,aplikasi,bahasa,prob_bahasa,rating,teks
441010,gopay,en,0.936,5,cakep dah
455437,gopay,en,0.305,5,ok bagus
168466,dana,en,0.174,5,mantap dana
414052,gopay,en,0.290,5,mantap lancar jaya
219689,dana,en,0.513,5,"simple, bgus,aman"
288271,dana,en,0.210,4,coba dulu
422536,gopay,en,0.342,1,gw mau narek uang gww wooii...
290424,dana,en,0.700,5,"dana is the best lah ok,,,"
334787,dana,ms,0.344,2,"kok duit aku kepotong yh, di notif emang bener jumlah yg masuk saat di top up. tp di saldo kok kurang yh 😔"
461651,gopay,ms,0.504,5,mudah untuk topap breee


### Keputusan 2.3 dan 2.4

## 2.5 Filter bahasa

In [21]:
BAHASA_DIPERTAHANKAN = {'id', 'ms'}
PROB_MIN_BAHASA_LAIN = 0.9
KATA_MIN_DETEKSI = 5

bukan_indonesia = (
    ~df['bahasa'].isin(BAHASA_DIPERTAHANKAN)
    & (df['prob_bahasa'] >= PROB_MIN_BAHASA_LAIN)
    & (df['jumlah_kata'] >= KATA_MIN_DETEKSI)
)
print('Jumlah yang dibuang per aplikasi:', bukan_indonesia.groupby(df['aplikasi']).sum().to_dict())
print('Label bahasa yang dibuang:', df.loc[bukan_indonesia, 'bahasa'].value_counts().to_dict())

contoh = df[bukan_indonesia].sample(min(int(bukan_indonesia.sum()), 20), random_state=SEED)
print('Contoh ulasan yang dibuang (cek apakah memang bukan Bahasa Indonesia):')
with pd.option_context('display.max_colwidth', 150):
    display(contoh.assign(teks=contoh['teks'].map(samarkan).str.slice(0, 150))[
        ['aplikasi', 'bahasa', 'prob_bahasa', 'rating', 'teks']
    ])

df = df[~bukan_indonesia].copy()
catat('bahasa Indonesia', df)
pd.DataFrame(jejak).set_index('langkah')

Jumlah yang dibuang per aplikasi: {'dana': 0, 'gopay': 0, 'ovo': 0}
Label bahasa yang dibuang: {}
Contoh ulasan yang dibuang (cek apakah memang bukan Bahasa Indonesia):


,aplikasi,bahasa,prob_bahasa,rating,teks


,dana,gopay,ovo,total
langkah,,,,
raw,347500,205000,27000,579500
dalam periode,326378,194383,25591,546352
tanpa duplikat id,326378,194383,25591,546352
teks tidak kosong,326378,194383,25591,546352
>= 2 kata,240075,136038,22890,399003
bahasa Indonesia,239965,135899,22878,398742
bahasa Indonesia,239965,135899,22878,398742
bahasa Indonesia,239965,135899,22878,398742


## 2.6 Simpan data bersih

In [22]:
df['bulan'] = df['tanggal'].dt.to_period('M').astype(str)

print('Jumlah ulasan bersih per aplikasi:', df.groupby('aplikasi').size().to_dict())
print('Distribusi rating (%):')
display(pd.crosstab(df['aplikasi'], df['rating'], normalize='index').mul(100).round(1))
print('Jumlah ulasan per bulan:')
display(pd.crosstab(df['bulan'], df['aplikasi']))

KOLOM_SIMPAN = KOLOM_STANDAR + ['jumlah_kata', 'bulan', 'hari_lonjakan', 'bahasa', 'prob_bahasa']
df[KOLOM_SIMPAN].to_parquet(FOLDER_PROCESSED / 'ulasan_bersih.parquet', index=False)
print(f'Tersimpan: ulasan_bersih.parquet ({len(df):,} baris)')

Jumlah ulasan bersih per aplikasi: {'dana': 239965, 'gopay': 135899, 'ovo': 22878}
Distribusi rating (%):


rating,1,2,3,4,5
aplikasi,,,,,
dana,27.5,5.0,5.3,6.0,56.2
gopay,19.9,3.1,3.7,6.3,66.9
ovo,67.0,6.8,4.9,2.9,18.5


Jumlah ulasan per bulan:


aplikasi,dana,gopay,ovo
bulan,,,
2025-10,18047,11945,2356
2025-11,17452,11090,2020
2025-12,20392,11194,2056
2026-01,24213,12336,2473
2026-02,19787,9403,1725
2026-03,21581,11251,1527
2026-04,15679,9133,1874
2026-05,16739,10091,1680
2026-06,16266,11297,1627


Tersimpan: ulasan_bersih.parquet (398,742 baris)


## 2.7 Kamus slang: bahan peninjauan

In [23]:
import urllib.request
from collections import Counter

FOLDER_KAMUS = FOLDER_PROJECT / 'kamus'
FOLDER_KAMUS.mkdir(parents=True, exist_ok=True)

# Kamus alay publik: Salsabila dkk. (2018), Colloquial Indonesian Lexicon
PATH_KAMUS_ALAY = FOLDER_KAMUS / 'kamus_alay_salsabila2018.csv'
if not PATH_KAMUS_ALAY.exists():
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv',
        PATH_KAMUS_ALAY,
    )
kamus_alay = pd.read_csv(PATH_KAMUS_ALAY)
kamus_alay['slang'] = kamus_alay['slang'].astype(str).str.lower().str.strip()
kamus_alay['formal'] = kamus_alay['formal'].astype(str).str.lower().str.strip()
peta_alay = kamus_alay.drop_duplicates('slang').set_index('slang')['formal'].to_dict()
kata_formal = set(kamus_alay['formal'])
print(f'Kamus alay: {len(peta_alay):,} entri')


def token_dasar(teks):
    """Huruf kecil, ambil kata alfabet, huruf berulang >= 3 dipendekkan (mantappp -> mantap)."""
    return [re.sub(r'(.)\1{2,}', r'\1', t) for t in re.findall(r'[a-z]+', teks.lower())]


frekuensi = Counter(t for teks in df['teks'] for t in token_dasar(teks))
total_token = sum(frekuensi.values())
token_diubah = sum(n for t, n in frekuensi.items() if peta_alay.get(t, t) != t)
print(f'Token di data bersih: {total_token:,} ({len(frekuensi):,} unik); '
      f'diubah oleh kamus alay: {token_diubah / total_token * 100:.1f}%')

print('\nA. 150 token tersering yang DIUBAH kamus alay (token->hasil (frekuensi)), cek pemetaan yang keliru:')
diubah = [(t, n) for t, n in frekuensi.most_common() if peta_alay.get(t, t) != t]
print('; '.join(f'{t}->{peta_alay[t]} ({n})' for t, n in diubah[:150]))

print('\nB. 250 token tersering yang TIDAK ADA di kamus alay (kandidat slang baru):')
tidak_dikenal = [(t, n) for t, n in frekuensi.most_common()
                 if t not in peta_alay and t not in kata_formal and len(t) > 1]
print(', '.join(f'{t} ({n})' for t, n in tidak_dikenal[:250]))

Kamus alay: 4,330 entri
Token di data bersih: 4,165,397 (56,037 unik); diubah oleh kamus alay: 10.8%

A. 150 token tersering yang DIUBAH kamus alay (token->hasil (frekuensi)), cek pemetaan yang keliru:
gak->enggak (33801); ga->enggak (31160); udah->sudah (20949); yg->yang (19128); aja->saja (17798); pake->pakai (14644); gk->enggak (13981); gimana->bagaimana (7413); udh->sudah (6958); gw->gue (5470); gabisa->enggak bisa (5401); d->ada (5017); minta->meminta (4539); bgt->banget (4482); karna->karena (4412); k->kak (3945); sy->saya (3866); dah->deh (3783); ilang->hilang (3594); rb->ribu (3565); trus->terus (3495); ni->ini (3420); nunggu->menunggu (3388); tp->tapi (3335); g->enggak (3301); tdk->tidak (3089); sampe->sampai (3079); ko->kok (3076); klo->kalo (3058); x->kali (2968); sya->saya (2926); nggak->enggak (2919); bener->benar (2877); tpi->tapi (2839); ngga->enggak (2749); gitu->begitu (2482); knp->kenapa (2480); gini->begini (2387); si->sih (2320); kaya->kayak (2267); tetep->tetap (22

### Hasil tinjauan kamus

## 2.8 Kamus final dan dua versi teks

In [24]:
!pip install -q Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 1.9 MB/s eta 0:00:0000:0100:01m


In [27]:
# Koreksi pemetaan kamus alay yang keliru untuk domain e-wallet
KOREKSI = {
    'gw': 'saya', 'gue': 'saya', 'gua': 'saya',
    'dah': 'sudah', 'dh': 'sudah', 'd': 'di',
    'klo': 'kalau', 'kalo': 'kalau',
    'narik': 'tarik',  # bukan "menarik" (interesting)
    'ngasih': 'kasih',
    'ngapain': 'mengapa',
    'kaga': 'tidak', 'kagak': 'tidak',
    'pengen': 'ingin',
}
# Pemetaan yang dibuang karena ambigu atau merusak istilah domain
HAPUS = {'bca', 'in', 'it', 's', 'si'}
# Slang dan varian baru dari daftar B
TAMBAHAN = {
    'apk': 'aplikasi', 'apl': 'aplikasi', 'app': 'aplikasi',
    'tf': 'transfer', 'trf': 'transfer', 'tranfer': 'transfer',
    'topap': 'topup',
    'eror': 'error', 'erorr': 'error', 'ganguan': 'gangguan',
    'mulu': 'terus', 'lemot': 'lambat', 'lelet': 'lambat', 'muter': 'berputar',
    'min': 'admin', 'duit': 'uang', 'nomer': 'nomor',
    'terimakasih': 'terima kasih', 'trimakasih': 'terima kasih',
    'kepotong': 'terpotong', 'membatu': 'membantu', 'mmbatu': 'membantu', 'mmbantu': 'membantu',
    'diana': 'dana', 'pinjem': 'pinjam', 'minjem': 'pinjam',
    'benerin': 'perbaiki', 'balikin': 'kembalikan', 'knapa': 'kenapa',
    'notif': 'notifikasi', 'dikit': 'sedikit',
    'coin': 'koin', 'point': 'poin', 'ok': 'oke', 'okey': 'oke', 'jos': 'mantap',
}
# Frasa yang disatukan sebelum pemetaan per kata
ATURAN_FRASA = [
    (r'\bgo\s*-?\s*pay\b', 'gopay'),
    (r'\btop\s*-?\s*up\b', 'topup'),
    (r'\bpay\s*-?\s*later\b', 'paylater'),
    (r'\bcustomer\s+service\b', 'cs'),
    (r'\be\s*-?\s*wallet\b', 'ewallet'),
]

kamus_final = {s: re.sub(r'\benggak\b', 'tidak', f) for s, f in peta_alay.items() if s not in HAPUS}
kamus_final['enggak'] = 'tidak'
kamus_final.update(KOREKSI)
kamus_final.update(TAMBAHAN)
kamus_final = {s: f for s, f in kamus_final.items() if s != f}

pd.DataFrame(sorted(kamus_final.items()), columns=['slang', 'formal']).to_csv(
    FOLDER_KAMUS / 'slang_final.csv', index=False
)
print(f'Kamus final: {len(kamus_final):,} entri, tersimpan di kamus/slang_final.csv')

Kamus final: 4,366 entri, tersimpan di kamus/slang_final.csv


In [28]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

KATA_DIPERTAHANKAN = {
    'tidak', 'belum', 'bukan', 'jangan', 'tanpa', 'kurang', 'tak',  # negasi
    'bisa', 'tolong', 'kenapa', 'masih',  # penanda keluhan
}
STOPWORD = (set(StopWordRemoverFactory().get_stop_words()) | {'nya'}) - KATA_DIPERTAHANKAN
(FOLDER_KAMUS / 'stopword.txt').write_text('\n'.join(sorted(STOPWORD)), encoding='utf-8')

POLA_URL = re.compile(r'https?://\S+|www\.\S+')
POLA_EMAIL = re.compile(r'\S+@\S+')
POLA_ANGKA_PANJANG = re.compile(r'\d{6,}')


def normalisasi(teks):
    """Teks -> daftar token ternormalisasi (huruf kecil, frasa disatukan, slang dipetakan)."""
    teks = teks.lower()
    teks = POLA_URL.sub(' ', teks)
    teks = POLA_EMAIL.sub(' email ', teks)
    teks = POLA_ANGKA_PANJANG.sub(' nomor ', teks)
    for pola, ganti in ATURAN_FRASA:
        teks = re.sub(pola, ganti, teks)
    teks = re.sub(r'([a-z])\1{2,}', r'\1', teks)  # mantappp -> mantap
    token = []
    for t in re.findall(r'[a-z0-9]+', teks):
        token.extend(kamus_final.get(t, t).split())
    return token


token_ringan = [normalisasi(t) for t in df['teks']]
df['teks_ringan'] = [' '.join(t) for t in token_ringan]
df['teks_penuh'] = [' '.join(w for w in t if w not in STOPWORD) for t in token_ringan]

print(f'Stopword: {len(STOPWORD)} kata (tanpa {sorted(KATA_DIPERTAHANKAN)})')
print('Ulasan dengan teks_penuh kosong:', int((df['teks_penuh'] == '').sum()))

contoh = df.sample(8, random_state=SEED)
print('Contoh hasil normalisasi:')
with pd.option_context('display.max_colwidth', 120):
    display(pd.DataFrame({
        'teks': contoh['teks'].map(samarkan).str.slice(0, 120),
        'teks_ringan': contoh['teks_ringan'].str.slice(0, 120),
        'teks_penuh': contoh['teks_penuh'].str.slice(0, 120),
    }))

KOLOM_SIMPAN = KOLOM_STANDAR + [
    'teks_ringan', 'teks_penuh', 'jumlah_kata', 'bulan', 'hari_lonjakan', 'bahasa', 'prob_bahasa'
]
df[KOLOM_SIMPAN].to_parquet(FOLDER_PROCESSED / 'ulasan_bersih.parquet', index=False)
print(f'Tersimpan: ulasan_bersih.parquet ({len(df):,} baris, dengan teks_ringan dan teks_penuh)')
pd.DataFrame(jejak).set_index('langkah')

Stopword: 117 kata (tanpa ['belum', 'bisa', 'bukan', 'jangan', 'kenapa', 'kurang', 'masih', 'tak', 'tanpa', 'tidak', 'tolong'])
Ulasan dengan teks_penuh kosong: 335
Contoh hasil normalisasi:


,teks,teks_ringan,teks_penuh
448852,sangat bagus,sangat bagus,sangat bagus
122282,kenapa akun dana saya tidak bisa di buka sampai2 di suspen segala,kenapa akun dana saya tidak bisa di buka sampai-sampai di suspen segala,kenapa akun dana tidak bisa buka sampai-sampai suspen segala
488659,"awal puasa masih bisa pinjem gopay pinjam beberapa kali, giliran mau lebaran malah di off,,,",awal puasa masih bisa pinjam gopay pinjam beberapa kali giliran mau lebaran malah di off,awal puasa masih bisa pinjam gopay pinjam beberapa kali giliran mau lebaran malah off
566589,pinjaman ada ta ????,pinjaman ada ta,pinjaman ta
514158,cukup mmbatu,cukup membantu,cukup membantu
173545,cukup memuaskan,cukup memuaskan,cukup memuaskan
277690,setiap d pindah saldo nya selalu d kenakan biaya,setiap di pindah saldo nya selalu di kenakan biaya,pindah saldo selalu kenakan biaya
265130,"Dana instan belum muncul,, tolong dimunculkan",dana instan belum muncul tolong dimunculkan,dana instan belum muncul tolong dimunculkan


Tersimpan: ulasan_bersih.parquet (398,742 baris, dengan teks_ringan dan teks_penuh)


,dana,gopay,ovo,total
langkah,,,,
raw,347500,205000,27000,579500
dalam periode,326378,194383,25591,546352
tanpa duplikat id,326378,194383,25591,546352
teks tidak kosong,326378,194383,25591,546352
>= 2 kata,240075,136038,22890,399003
bahasa Indonesia,239965,135899,22878,398742
bahasa Indonesia,239965,135899,22878,398742
bahasa Indonesia,239965,135899,22878,398742
